##  Imports and utility

In [ ]:
from schema import common_mice
from schema.mpanze_paw_tracking_refactor import mpanze_paw_tracking_refactor as pt
from schema.mpanze_exp_refactor import mpanze_exp_refactor as exp
from schema.mpanze_widefield_refactor import mpanze_widefield_refactor as wf
import numpy as np
from pathlib import Path
import pandas as pd
from datetime import datetime
from tqdm.autonotebook import tqdm
from mpanze_scripts.util.allen_utils import load_allen, overlay_allen

import warnings
import matplotlib.pyplot as plt
from matplotlib import rc
import seaborn as sns
%matplotlib inline
import cv2

import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from statannotations.Annotator import Annotator, PValueFormat

from scipy.ndimage import gaussian_filter1d

base = importr('base')
lme4 = importr('lme4')
emmeans = importr('emmeans')
stats = importr('stats')

# define path for datasets
# p_datasets = Path('~/neurophys_3/r_outputs/datasets/').expanduser()
# p_datasets.mkdir(parents=True, exist_ok=True)
p_figures = Path('~/neurophys_3/r_outputs/figures/figure_2/').expanduser()
p_figures.mkdir(parents=True, exist_ok=True)

Figure definitions - manuscript

In [ ]:
# set font to Arial
rc('font',**{'family':'sans-serif','sans-serif':['Arial']})
# set font sizes to 12 for figures
rc('font', size=12)          # controls default text sizes
rc('axes', titlesize=12)     # fontsize of the axes title
rc('axes', labelsize=12)    # fontsize of the x and y labels
rc('xtick', labelsize=12)    # fontsize of the tick labels
rc('ytick', labelsize=12)    # fontsize of the tick labels
rc('legend', fontsize=10)    # legend fontsize
rc('figure', titlesize=12)  # fontsize of the figure title

# set line width to 1
rc('lines', linewidth=1)

# set dpi to 600 for figures
rc('figure', dpi=600)

# svg font type shenanigans
rc('svg', fonttype='none')

fontsize_small = 10
fontsize_medium = 12
fontsize_large = 14

# define conversion factor from inches to cm for convenience
cm = 1/2.54 * 1.5 # (scale larger for Manuscript)

# color palette for cohorts
group_colors = {'Sham':'#BBBBBB', 'Stroke':'#4477AA', 'Stroke + training':'#AA3377'}

## Figure 2B-C - expert maps

### load dataset

In [ ]:
wf_param_id = 6
window = np.arange(-100, 101)  # -5 to 5 s around grasp
frames_map = np.array([-10, -5, 0, 5, 10, 15, 20, 25, 30])
example_rois = ["MOp_contra", "MOs-lateral_contra", "MOp_ipsi"]

keys = (
    wf.ImageProcessing2
    * exp.ExperimentalPhase
    * exp.DaysFromStrokeNorm
    * exp.StrokeGroup.proj(stroke_group='group')
    & f"wf_param_id={wf_param_id}"
    & "mouse_id > 40"
    & [f"days_from_stroke_norm={d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "phase = 'Expert'"
    & "stroke_group != 'Learning'"
).fetch("KEY")
print(len(keys), "sessions to process")

index = []
rows_wf = []
rows_rois = []
for key in tqdm(keys, desc="Processing sessions"):
    # fetch widefield data
    u, svt, h, w = (wf.ImageProcessing2 & key).load_components(svt_baseline=True)
    t_wf = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")
    M = (wf.RescaledAllenRegistration2 & key).fetch1("allen_matrix_rescaled")
    handedness = (exp.Handedness & key).fetch1("handedness")

    # get paw data
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1('KEY')
    start_times, epoch_ids = (
        pt.MovementSegmentation.Epoch
        * pt.EpochClassification.Epoch
        & dict(**key_ipsi, epoch_class='rewarded')
    ).fetch('start_time', 'epoch_id')

    # get ROI responses
    roi_dffs, roi_ids = (wf.AllenSegmentation2 & key).load_rois(example_rois)
    # roi_dffs, roi_ids = (wf.AllenSegmentation2.ROI & key & [f"roi_id='{r}'" for r in example_rois]).fetch("dff", "roi_id", order_by="roi_id")

    # iterate over epochs
    d = (exp.DaysFromStrokeNorm & key).fetch1('days_from_stroke_norm')
    for start_time, epoch_id in zip(start_times, epoch_ids):
        # process maps
        start_frame = np.searchsorted(t_wf, start_time)

        # check bounds
        if start_frame + window[0] < 0 or start_frame + window[-1] >= len(t_wf):
            continue

        svt_window = svt[:, start_frame + frames_map]
        map_window = (u @ svt_window).T.reshape(-1, h, w)
        # warp and flip
        for i in range(map_window.shape[0]):
            map_window[i] = cv2.warpAffine(map_window[i], M, (h,w))
            if handedness == 'R':
                map_window[i] = np.fliplr(map_window[i])
        index.append((key['mouse_id'], d, epoch_id))
        rows_wf.append([map_window.astype(np.float32) * 100])

        # process ROIs
        row_roi = []
        for roi_dff in roi_dffs:
            roi_window = roi_dff[start_frame + window]
            row_roi.append(roi_window.astype(np.float32) * 100)
        rows_rois.append(row_roi)

index = pd.MultiIndex.from_tuples(index, names=['mouse_id', 'day', 'epoch_id'])
df_expert_maps = pd.DataFrame(rows_wf, index=index, columns=['wf_maps'])
df_expert_rois = pd.DataFrame(rows_rois, index=index, columns=roi_ids)


In [ ]:
# save datasets
df_expert_maps.to_pickle(p_figures / f'figure_2_expert_response_maps_wf{wf_param_id}.pkl')
df_expert_rois.to_pickle(p_figures / f'figure_2_expert_response_rois_wf{wf_param_id}.pkl')

In [ ]:
# load datasets
wf_param_id = 6
df_expert_maps = pd.read_pickle(p_figures / f'figure_2_expert_response_maps_wf{wf_param_id}.pkl')
df_expert_rois = pd.read_pickle(p_figures / f'figure_2_expert_response_rois_wf{wf_param_id}.pkl')

In [ ]:
# count grasps
sum(df_expert_rois.groupby("mouse_id").count()["MOp_contra"])

### plot maps

In [ ]:
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl",
]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
#areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255


f, ax = plt.subplots(1, 9, figsize=(12.5*cm, 2*cm), edgecolor='k',
                     gridspec_kw=dict(wspace=0,hspace=0,top=1,bottom=0,left=0.07,right=1))
cax = f.add_axes([0.01, 0.2, 0.02, 0.6])
vmin = 0
vmax = 0.8
timestamps = [-0.5, -0.25, 0, 0.25, 0.5, 0.75, 1, 1.25, 1.5]
dff_stack = np.stack(df_expert_maps['wf_maps'].to_numpy())
mean_dff = np.nanmean(dff_stack, axis=0)
for i in range(9):
    img = mean_dff[i]
    img[mask_combined==0] = np.nan

    imsh = ax[i].imshow(img, vmin=vmin, vmax=vmax, cmap='viridis')
    ax[i].axis('off')
    overlay_allen(ax[i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='w', linewidth=0.5, alpha=0.5))

    ax[i].text(0.5, 0.85, f'{timestamps[i]:.2f} s', color='k', fontsize=fontsize_small, ha='center', va='bottom', transform=ax[i].transAxes)
    ax[i].set_xlim(10,118)
plt.colorbar(imsh, cax=cax)
cax.set_ylabel('$\Delta$F/F (%)', fontsize=fontsize_small)
cax.set_yticks([])
cax.text(0.5, -0.05, f'{vmin}', fontsize=fontsize_small, ha='center', va='top', transform=cax.transAxes)
cax.text(0.5, 1.05, f'{vmax}', fontsize=fontsize_small, ha='center', va='bottom', transform=cax.transAxes)
f.savefig(p_figures / f'figure_2_widefield_maps_expert_wf{wf_param_id}.svg', dpi=600, transparent=True)
plt.show()

### plot ROIs

In [ ]:
nanmean = lambda x: np.nanmean(np.stack(x), axis=0)
window = np.arange(-100, 101)  # -5 to 5 s around grasp
t = window / 20
f, ax = plt.subplots(1, 1, figsize=(4*cm, 3*cm), edgecolor='k',
                     gridspec_kw=dict(wspace=0,hspace=0,top=0.95,bottom=0.31,left=0.3,right=0.95))
example_rois = ["MOp_contra", "MOs-lateral_contra", "MOp_ipsi"]
for roi in example_rois:
    dff = np.stack(df_expert_rois[roi].to_numpy())
    mean_dff = np.nanmean(dff, axis=0)
    sem_dff = np.nanstd(dff, axis=0) / np.sqrt(dff.shape[0])
    
    # sem_dff = np.nanstd(dff, axis=0) / np.sqrt(dff.shape[0])
    ax.plot(t, mean_dff, label=roi, lw=1)
    ax.fill_between(t, mean_dff-sem_dff, mean_dff+sem_dff, alpha=0.3, ec='None')
# ax.legend()
# set spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(-5, 5)
ax.set_ylabel('$\Delta$F/F (%)')
ax.set_yticks([0, 0.8])
# add inset axes
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl"
]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255

ax_inset = f.add_axes([0.75, 0.75, 0.25, 0.25], zorder=-1)
overlay_allen(ax_inset, areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
              line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
ax_inset.set_aspect('equal')
ax_inset.set_ylim(128,0)
# create colored masks
tab10 = plt.get_cmap('tab10', 10)
mask_rgb = np.full((128, 128, 3), 1.0, dtype=np.float32)
for i, roi in enumerate(example_rois):
    area_to_use = roi.replace('_ipsi', '_L').replace('_contra', '_R')
    mask_roi = masks[area_names == area_to_use].squeeze()
    color = tab10(i % 10)[:3]
    mask_rgb[mask_roi > 0] = color
ax_inset.imshow(mask_rgb)
ax.patch.set_visible(False)
ax.set_xlabel('Time from grasp (s)')
ax.axvspan(0,1,color='gray', alpha=0.2, ec='None')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.5)
plt.show()
f.savefig(p_figures / f'figure_2_roi_traces_expert_wf{wf_param_id}.svg', dpi=600, transparent=True)

## Figure 2D-E

### load data

In [ ]:
wf_param_id = 6
response_window = np.arange(0, 20)  # 0 to 1 s around grasp
window_cont = np.arange(-100, 101) # -5 to 5 s around grasp (for continuous traces)
rois = [
    "MOs-lateral", "MOs-medial", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSp-anterior", "RSP-posterior", "VISp", "VIS-medial", "VISa", "VISrl"
]
rois_for_stats = [r + "_contra" for r in rois] + [r + "_ipsi" for r in rois]

rois_for_cont = ["MOs-lateral", "MOs-medial", "MOp", "SSp-ll", "SSp-ul"]
rois_for_cont = [r + "_contra" for r in rois_for_cont] + [r + "_ipsi" for r in rois_for_cont]

keys = (
    wf.ImageProcessing2
    * exp.ExperimentalPhase
    * exp.DaysFromStrokeNorm
    * exp.StrokeGroup.proj(stroke_group='group')
    & f"wf_param_id={wf_param_id}"
    & "mouse_id > 40"
    & [f"days_from_stroke_norm={d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
).fetch("KEY")
print(len(keys), "sessions to process")

index = []
rows_wf = []
rows_rois = []
rows_rois_cont = []
for key in tqdm(keys, desc="Processing sessions"):
    # fetch widefield data
    u, svt, h, w = (wf.ImageProcessing2 & key).load_components(svt_baseline=True)
    t_wf = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")
    M = (wf.RescaledAllenRegistration2 & key).fetch1("allen_matrix_rescaled")
    handedness = (exp.Handedness & key).fetch1("handedness")

    # get paw data
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1('KEY')
    start_times, epoch_ids = (
        pt.MovementSegmentation.Epoch
        * pt.EpochClassification.Epoch
        & dict(**key_ipsi, epoch_class='rewarded')
    ).fetch('start_time', 'epoch_id')

    # get ROI dff
    roi_dffs, roi_ids = (wf.AllenSegmentation2 & key).load_rois(rois_for_stats)
    roi_dffs_cont, roi_ids_cont = (wf.AllenSegmentation2 & key).load_rois(rois_for_cont)

    # iterate over epochs
    d = (exp.DaysFromStrokeNorm & key).fetch1('days_from_stroke_norm')
    for start_time, epoch_id in zip(start_times, epoch_ids):
        # process maps
        start_frame = np.searchsorted(t_wf, start_time)

        # check bounds
        if start_frame + window_cont[0] < 0 or start_frame + window_cont[-1] >= len(t_wf):
            continue

        svt_window = svt[:, start_frame + response_window].mean(axis=1).reshape(-1,1)
        map_window = (u @ svt_window).T.reshape(h, w)
        # warp and flip
        map_window = cv2.warpAffine(map_window, M, (h,w))
        if handedness == 'R':
            map_window = np.fliplr(map_window)
        index.append((key['mouse_id'], d, epoch_id))
        rows_wf.append([map_window.astype(np.float32) * 100])

        # process ROIs
        row_roi = []
        for roi_dff in roi_dffs:
            roi_window = roi_dff[start_frame + response_window].mean()
            row_roi.append(roi_window * 100)
        rows_rois.append(row_roi)
        row_roi_cont = []
        for roi_dff in roi_dffs_cont:
            roi_window_cont = roi_dff[start_frame + window_cont]
            row_roi_cont.append(roi_window_cont.astype(np.float32) * 100)
        rows_rois_cont.append(row_roi_cont)

index = pd.MultiIndex.from_tuples(index, names=['mouse_id', 'day', 'epoch_id'])
df_response_maps = pd.DataFrame(rows_wf, index=index, columns=['wf_maps'])
df_response_rois = pd.DataFrame(rows_rois, index=index, columns=roi_ids)
df_response_rois_cont = pd.DataFrame(rows_rois_cont, index=index, columns=roi_ids_cont)

dataset_info = (
    (
        exp.StrokeGroup.proj(stroke_group='group')
        * exp.ExperimentalPhase
        * exp.DaysFromStrokeNorm
        & keys
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "phase", "stroke_group"])
    .rename(columns={"days_from_stroke_norm": "day"})
    .set_index(["mouse_id", "day"])        
)
# rename groups
dataset_info["stroke_group"] = dataset_info["stroke_group"].replace({"Rehab": "Stroke + training"})
dataset_info["phase"] = dataset_info["phase"].replace({"Expert":"Pre", "Early":"Post Early", "Late":"Post Late"})
# create categorical variables - they work better with R for stats
dataset_info["p"] = pd.Categorical(dataset_info["phase"], categories=["Pre", "Post Early", "Post Late"], ordered=True)
dataset_info["g"] = pd.Categorical(dataset_info["stroke_group"], categories=["Sham", "Stroke", "Stroke + training"], ordered=True)
# day catergorical variable should be converted to string
dataset_info["d"] = pd.Categorical(dataset_info.index.get_level_values("day").astype(str), categories=[str(d) for d in [-3, -2, -1, 3, 7, 14, 21, 28]], ordered=True)
display(df_response_maps.head())
display(df_response_rois.head())
display(dataset_info.head())

In [ ]:
# save datasets
df_response_maps.to_pickle(p_figures / f'figure_2_response_maps_wf{wf_param_id}.pkl')
df_response_rois.to_pickle(p_figures / f'figure_2_response_rois_wf{wf_param_id}.pkl')
df_response_rois_cont.to_pickle(p_figures / f'figure_2_response_rois_cont_wf{wf_param_id}.pkl')
dataset_info.to_pickle(p_figures / f'figure_2_dataset_info_wf{wf_param_id}.pkl')

In [ ]:
# load datasets
wf_param_id = 6
df_response_maps = pd.read_pickle(p_figures / f'figure_2_response_maps_wf{wf_param_id}.pkl')
df_response_rois = pd.read_pickle(p_figures / f'figure_2_response_rois_wf{wf_param_id}.pkl')
df_response_rois_cont = pd.read_pickle(p_figures / f'figure_2_response_rois_cont_wf{wf_param_id}.pkl')
dataset_info = pd.read_pickle(p_figures / f'figure_2_dataset_info_wf{wf_param_id}.pkl')

### prepare data for plotting

In [ ]:
nanmean = lambda x: np.nanmean(np.stack(x, axis=0), axis=0)
nanmedian = lambda x: np.nanmedian(np.stack(x, axis=0), axis=0)

df_response_maps["wf_maps_baselined"] = df_response_maps['wf_maps'] - df_response_maps.query("day<0").groupby("mouse_id")['wf_maps'].agg(nanmedian)

responses_by_phase_group = (
    df_response_maps
    .join(dataset_info)
    .groupby(['p', 'g'])
    .agg(dict(wf_maps_baselined=nanmean))
)

df_response_rois_for_stats = df_response_rois.stack().to_frame(name='response')
df_response_rois_for_stats.index.names = ['mouse_id', 'day', 'epoch_id', 'roi']
df_response_rois_for_stats["y"] = df_response_rois_for_stats["response"] - df_response_rois_for_stats.query("day<0").groupby(["mouse_id", "roi"])["response"].median()
df_response_rois_for_stats

# run stats for each ROI
results = []
for roi, df_roi in tqdm(df_response_rois_for_stats.join(dataset_info).query("p!='Pre'").groupby('roi')):
    with (robjects.default_converter + pandas2ri.converter).context():
        model = lme4.lmer('y ~ 1 + g * p + (1|mouse_id/d)', data=df_roi.reset_index())
        formula = "pairwise ~ g | p"
        emm = emmeans.emmeans(model, stats.formula(formula), adjust="none")
        contrasts = base.summary(emm[1])
        confints = stats.confint(emm[1])
    
    # add confints to dataframe
    contrasts = contrasts.set_index(["contrast", "p"])
    confints = confints.set_index(["contrast", "p"]).filter(["asymp.LCL", "asymp.UCL"])
    contrasts = contrasts.join(confints).reset_index()
    contrasts["roi_name"] = roi
    results.append(contrasts)
from scipy.stats import false_discovery_control
df_results_roi = pd.concat(results, ignore_index=True)
df_results_roi["p_adj"] = false_discovery_control(df_results_roi["p.value"], method='bh')
df_results_roi["comparison"] = [f"({row['roi_name']}) {row['p']}: {row['contrast']}" for _, row in df_results_roi.iterrows()]
df_results_roi.to_csv(p_figures / f"figure_2_roi_response_stats_wf{wf_param_id}.csv")

In [ ]:
df_results_roi.query("roi_name=='MOp_ipsi' and p=='Post Late'")

### Figure 2D - Example ROIs

In [ ]:
nanmedian_full = lambda x: np.nanmedian(np.stack(x, axis=0)[20:40], axis=(0,1))
window_cont = np.arange(-100, 101) # -5 to 5 s around grasp (for continuous traces)

stack_mean = lambda x: np.mean(np.stack(x, axis=0), axis=0)
stack_sem = lambda x: np.std(np.stack(x, axis=0), axis=0) / np.sqrt(len(x))

roi_to_plot = "MOp_contra"
roi_to_plot_baselines = df_response_rois_cont.query("day<0").groupby("mouse_id")[roi_to_plot].agg(nanmedian_full)
roi_to_plot_copy = df_response_rois_cont[roi_to_plot] # - roi_to_plot_baselines
mean_by_p_g = roi_to_plot_copy.to_frame().join(dataset_info).groupby(['p', 'g']).agg({roi_to_plot: stack_mean})
sem_by_p_g = roi_to_plot_copy.to_frame().join(dataset_info).groupby(['p', 'g']).agg({roi_to_plot: stack_sem})


f, ax = plt.subplots(1, 3, figsize=(10*cm, 3.5*cm),
                     gridspec_kw=dict(wspace=0.2, hspace=0, top=0.95, bottom=0.3, left=0.13, right=1))
t = window_cont / 20

# post early
phase = "Post Early"
mean_sham = mean_by_p_g.loc[(phase, "Sham"), roi_to_plot]
mean_stroke = mean_by_p_g.loc[(phase, "Stroke"), roi_to_plot]
mean_rehab = mean_by_p_g.loc[(phase, "Stroke + training"), roi_to_plot]
sem_sham = sem_by_p_g.loc[(phase, "Sham"), roi_to_plot]
sem_stroke = sem_by_p_g.loc[(phase, "Stroke"), roi_to_plot]
sem_rehab = sem_by_p_g.loc[(phase, "Stroke + training"), roi_to_plot]
ax[0].plot(t, mean_sham, label='Sham', color=group_colors['Sham'])
ax[0].fill_between(t, mean_sham - sem_sham, mean_sham + sem_sham, color=group_colors['Sham'], alpha=0.3, ec='None')
ax[0].plot(t, mean_stroke, label='Stroke', color=group_colors['Stroke'])
ax[0].fill_between(t, mean_stroke - sem_stroke, mean_stroke + sem_stroke, color=group_colors['Stroke'], alpha=0.3, ec='None')
ax[0].plot(t, mean_rehab, label='Stroke + training', color=group_colors['Stroke + training'])
ax[0].fill_between(t, mean_rehab - sem_rehab, mean_rehab + sem_rehab, color=group_colors['Stroke + training'], alpha=0.3, ec='None')
ax[0].axvspan(0,1,color='gray', alpha=0.2, ec='None')
ax[0].spines['top'].set_visible(False)
ax[0].spines['right'].set_visible(False)
ax[0].set_xlim(-5, 5)
ax[0].set_ylim(-0.5, 1)
# ax[0].set_title("Early")
ax[0].axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.5)
ax[0].set_ylabel('$\Delta$F/F (%)')
ax[0].set_xlabel('Time from grasp (s)')
ax[0].text(0.05, 1, "Early", fontsize=fontsize_medium, ha='left', va='top', transform=ax[0].transAxes)


# post late
phase = "Post Late"
mean_sham = mean_by_p_g.loc[(phase, "Sham"), roi_to_plot]
mean_stroke = mean_by_p_g.loc[(phase, "Stroke"), roi_to_plot]
mean_rehab = mean_by_p_g.loc[(phase, "Stroke + training"), roi_to_plot]
sem_sham = sem_by_p_g.loc[(phase, "Sham"), roi_to_plot]
sem_stroke = sem_by_p_g.loc[(phase, "Stroke"), roi_to_plot]
sem_rehab = sem_by_p_g.loc[(phase, "Stroke + training"), roi_to_plot]
ax[1].plot(t, mean_sham, label='Sham', color=group_colors['Sham'])
ax[1].fill_between(t, mean_sham - sem_sham, mean_sham + sem_sham, color=group_colors['Sham'], alpha=0.3, ec='None')
ax[1].plot(t, mean_stroke, label='Stroke', color=group_colors['Stroke'])
ax[1].fill_between(t, mean_stroke - sem_stroke, mean_stroke + sem_stroke, color=group_colors['Stroke'], alpha=0.3, ec='None')
ax[1].plot(t, mean_rehab, label='Stroke + training', color=group_colors['Stroke + training'])
ax[1].fill_between(t, mean_rehab - sem_rehab, mean_rehab + sem_rehab, color=group_colors['Stroke + training'], alpha=0.3, ec='None')
ax[1].axvspan(0,1,color='gray', alpha=0.2, ec='None')
ax[1].spines['top'].set_visible(False)
ax[1].spines['right'].set_visible(False)
ax[1].set_xlim(-5, 5)
ax[1].set_ylim(-0.5, 1)
ax[1].text(0.05, 1, "Late", fontsize=fontsize_medium, ha='left', va='top', transform=ax[1].transAxes)
ax[1].axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.5)
ax[1].set_yticklabels([])

# boxplot of responses
df_for_boxplot = df_response_rois_for_stats.query(f"roi == '{roi_to_plot}'").join(dataset_info).query("p!='Pre'").reset_index()
df_summary = df_for_boxplot.groupby(["p", "g"])["y"].agg(["count", "median", lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)]).reset_index()
df_summary.to_csv(p_figures / f'figure_2_roi_response_summary_{roi_to_plot}_wf{wf_param_id}.csv')
display(df_for_boxplot.head())
sns.boxplot(
    x='p',
    y='y',
    hue='g',
    data=df_for_boxplot,
    ax=ax[2],
    palette=group_colors,
    # hue_order=["Sham", "Stroke", "Stroke + training"],
    gap=0.2,
    fill=False,
    showfliers=False,
    legend=False,
    # annotate mean
    meanline=True,
    meanprops=dict(linestyle='--', linewidth=5, color='k')
)
# add statistical annotations
pairs = []
p_values = []
for i, row in df_results_roi.query("roi_name == @roi_to_plot").iterrows():
    g1, g2 = row['contrast'].split(" - ")
    if g1 == "(Stroke + training)":
        g1 = "Stroke + training"
    if g2 == "(Stroke + training)":
        g2 = "Stroke + training"
    pairs.append(((row['p'], g1), (row['p'], g2)))
    p_values.append(row['p_adj'])

# print(pairs)

p_values = np.array(p_values)

annotator = Annotator(ax[2], pairs, data=df_for_boxplot.reset_index(), x="p", y="y", hue="g", hue_order=["Sham", "Stroke", "Stroke + training"])
annotator.configure(test=None, text_format='full', loc='inside', verbose=0, hide_non_significant=True,
                    fontsize=fontsize_small, line_width=1, text_offset=0, line_offset=0, use_fixed_offset=True)
annotator.set_pvalues(p_values)
annotator.annotate()


sns.stripplot(
    x='p',
    y='y',
    hue='g',
    # hue_order=["Sham", "Stroke", "Stroke + training"],
    data=df_for_boxplot,
    ax=ax[2],
    palette=group_colors,
    dodge=True,
    size=2,
    alpha=0.02,
    legend=False,
)
# ax[2].set_ylim(-0.2, 0.3)
ax[2].axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.5)
ax[2].set_xticks([0, 1, 2], labels=['Pre', 'Early', 'Late'])
ax[2].set_xlabel("")
ax[2].spines['top'].set_visible(False)
ax[2].spines['right'].set_visible(False)
# ax[2].set_title("Responses (- Pre)")
# ax[2].text(0.05, 0.8, "Responses\n rel. to Pre", fontsize=8, ha='left', va='bottom', transform=ax[2].transAxes)
ax[2].set_ylabel('')
ax[2].set_xlim([0.5, 2.5])
ax[2].set_xlabel('Phase')

ax_inset = f.add_axes([0.25, 0.75, 0.25, 0.25], zorder=5)
overlay_allen(ax_inset, areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
              line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
ax_inset.set_aspect('equal')
ax_inset.set_ylim(128,0)
# create colored masks
tab10 = plt.get_cmap('tab10', 10)
mask_rgb = np.full((128, 128, 3), 1.0, dtype=np.float32)
roi_for_mask = roi_to_plot.replace('_ipsi', '_L').replace('_contra', '_R')
mask_roi = masks[area_names == roi_for_mask].squeeze()
mask_rgb[mask_roi > 0] = 0.0
ax_inset.imshow(mask_rgb)

# f.tight_layout()
f.savefig(p_figures / f'figure_2_roi_trace_example_{roi_to_plot}_wf{wf_param_id}.svg', dpi=600, transparent=True)
plt.show()

# plt.figure()
# df_roi_response = 

### Figure 2E

In [ ]:
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl",
]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255


phase_to_ax={'Post Early':0, 'Post Late':1}
vmin = -0.5
vmax = 0.5
cmap = 'PiYG'
f, ax = plt.subplots(3,2, figsize=(9*cm, 10*cm), sharex=True, sharey=True,
                     gridspec_kw=dict(wspace=0, hspace=0.15, left=0, right=0.95, top=0.9, bottom=0.05))
for phase, df_phase in responses_by_phase_group.groupby("p"):
    if phase=='Pre':
        continue
    sham_vs_stroke = df_phase.loc[(phase, "Stroke"), "wf_maps_baselined"] - df_phase.loc[(phase, "Sham"), "wf_maps_baselined"]
    sham_vs_rehab = df_phase.loc[(phase, "Stroke + training"), "wf_maps_baselined"] - df_phase.loc[(phase, "Sham"), "wf_maps_baselined"]
    stroke_vs_rehab = df_phase.loc[(phase, "Stroke + training"), "wf_maps_baselined"] - df_phase.loc[(phase, "Stroke"), "wf_maps_baselined"]
    sham_vs_stroke[mask_combined==0] = np.nan
    sham_vs_rehab[mask_combined==0] = np.nan
    stroke_vs_rehab[mask_combined==0] = np.nan
    # plot
    i = phase_to_ax[phase]
    imsh0 = ax[0, i].imshow(sham_vs_stroke, vmin=vmin, vmax=vmax, cmap=cmap)
    imsh1 = ax[1, i].imshow(sham_vs_rehab, vmin=vmin, vmax=vmax, cmap=cmap)
    imsh2 = ax[2, i].imshow(stroke_vs_rehab, vmin=vmin, vmax=vmax, cmap=cmap)
    # allen
    overlay_allen(ax[0, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
    overlay_allen(ax[1, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
    overlay_allen(ax[2, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
    # titles
    ax[0,i].set_xlim(10,118)
    ax[1,i].set_xlim(10,118)
    ax[2,i].set_xlim(10,118)

    significant_rois = df_results_roi.query("p==@phase and p_adj<0.05")
    for j, row in significant_rois.iterrows():
        # replace _contra
        # replace _contra with _R and _ipsi with _L
        area_to_star = row['roi_name'].replace('_contra', '_R').replace('_ipsi', '_L')
        # get mask for area
        mask_roi = masks[area_names == area_to_star].squeeze()
        # get coordinates of center of mass
        from scipy.ndimage import center_of_mass
        com = center_of_mass(mask_roi)
        p_value = row['p_adj']
        if p_value < 0.001:
            star = '***'
        elif p_value <= 0.01:
            star = '**'
        elif p_value <= 0.05:
            star = '*'
        # contrast
        if row['contrast'] == "Sham - Stroke":
            ax[0,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')
        elif row['contrast'] == "Sham - (Stroke + training)":
            ax[1,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')
        elif row['contrast'] == "Stroke - (Stroke + training)":
            ax[2,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')

    # print(phase, significant_rois)


# ax[0,1].text(0.5, 0.95, "Stroke - Sham", fontsize=8, ha='center', va='center', transform=ax[0,1].transAxes)
# ax[1,1].text(0.5, 0.95, "(Stroke + training) - Sham", fontsize=8, ha='center', va='center', transform=ax[1,1].transAxes)
# ax[2,1].text(0.5, 0.95, "(Stroke + training) - Stroke", fontsize=8, ha='center', va='center', transform=ax[2,1].transAxes)

# pre, early, late
ax[-1,0].text(0.5, -0.1, "Early", fontsize=fontsize_medium, ha='center', va='bottom', transform=ax[-1,0].transAxes)
ax[-1,1].text(0.5, -0.1, "Late", fontsize=fontsize_medium, ha='center', va='bottom', transform=ax[-1,1].transAxes)
ax[0,0].text(0.5, 0.95, "Response differences between groups\nrelative to pre-stroke", fontsize=fontsize_medium, ha='center', va='top', transform=f.transFigure)

# colorbar text
cbar_0 = plt.colorbar(imsh0, ax=ax[0,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_0.ax.text(0.5, -0.07, "Sham > Stroke", fontsize=fontsize_small, ha='center', va='top', transform=cbar_0.ax.transAxes)
cbar_0.ax.text(0.5, 1.07, "Stroke > Sham", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_0.ax.transAxes)

cbar_1 = plt.colorbar(imsh0, ax=ax[1,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_1.ax.text(0.5, -0.07, "Sham > (Stroke + training)", fontsize=fontsize_small, ha='center', va='top', transform=cbar_1.ax.transAxes)
cbar_1.ax.text(0.5, 1.07, "(Stroke + training) > Sham", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_1.ax.transAxes)

cbar_2 = plt.colorbar(imsh0, ax=ax[2,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_2.ax.text(0.5, -0.07, "Stroke > (Stroke + training)", fontsize=fontsize_small, ha='center', va='top', transform=cbar_2.ax.transAxes)
cbar_2.ax.text(0.5, 1.07, "(Stroke + training) > Stroke", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_2.ax.transAxes)


f.savefig(p_figures / f'figure_2_widefield_response_differences_wf{wf_param_id}.svg', dpi=600, transparent=True)
plt.show()

## correlations

In [ ]:
dataset_lesion_size = (
    exp.StrokeVolume
    .fetch(format='frame')
    .reset_index()
    .filter(['mouse_id', 'stroke_volume'])
    .dropna()
    .set_index('mouse_id')
)
dataset_lesion_size

# get performance datasets
p_dataset_performance = Path('~/neurophys_3/r_outputs/datasets/dataset_performance_rates.pkl').expanduser()
dataset_performance = pd.read_pickle(p_dataset_performance)
dataset_performance_rel = dataset_performance - dataset_performance.query("day<0").groupby("mouse_id").median()

# get fine motor skill datasets
p_dataset_finemotor = Path('~/neurophys_3/r_outputs/datasets/df_onset_offset_ipsi.pkl').expanduser()
dataset_finemotor = pd.read_pickle(p_dataset_finemotor)
dataset_finemotor_rel = dataset_finemotor - dataset_finemotor.query("day<0").groupby("mouse_id").median()
dataset_finemotor_rel


### Supplementary figure 2C - lesion size

In [ ]:
roi_for_corr = "MOp_contra"
df_corr = (
    df_response_rois_for_stats
    .query(f"roi == '{roi_for_corr}'")
    .join(dataset_info).groupby(["mouse_id", "p"])
    .agg(dict(y='mean', g='first'))
    .join(dataset_lesion_size)
)
df_corr.query("p!='Pre'").dropna().to_csv(p_figures / f'figure_2_correlation_lesion_size_data_{roi_for_corr}_wf{wf_param_id}.csv')
display(df_corr)

f, ax = plt.subplots(1, 2, figsize=(5*cm, 3.5*cm), sharex=True, sharey=True,
                     gridspec_kw=dict(wspace=0, hspace=0, top=0.95, bottom=0.2, left=0.2, right=0.95))
early = df_corr.query("p == 'Post Early'").dropna()
colors_early = [group_colors[g] for g in early["g"]]
from scipy.stats import spearmanr
rho, pval = spearmanr(early["stroke_volume"], early['y'])
ax[0].scatter(early["stroke_volume"], early['y'], label='Early', c=colors_early, alpha=.7, edgecolors='none')
ax[0].text(0.03, 0.97, f"$\\rho$={rho:.2f}\np={pval:.2f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[0].transAxes)
late = df_corr.query("p == 'Post Late'").dropna()
colors_late = [group_colors[g] for g in late["g"]]
rho, pval = spearmanr(late["stroke_volume"], late['y'])
ax[1].scatter(late["stroke_volume"], late['y'], label='Late', c=colors_late, alpha=.7, edgecolors='none')
ax[1].text(0.03, 0.97, f"$\\rho$={rho:.2f}\np={pval:.2f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[1].transAxes)

ax[0].set_title("Early")
ax[1].set_title("Late")
ax[0].set_ylabel("Change in $\\Delta$F/F (%)")
ax[0].set_xlabel("Lesion volume (mm$^3$)")
f.savefig(p_figures / f'SF2_correlation_lesion_size_{roi_for_corr}_wf{wf_param_id}.svg', dpi=600, transparent=True)
plt.show()

### Figure 2F, Supplementary figure 2A - task performance

In [ ]:
roi_for_corr = "MOp_contra"
finemotor_for_corr = "rewarded_rate"
xlabel = "Change in rewarded rate (s$^{-1}$)"

df_corr = (
    df_response_rois_for_stats
    .query(f"roi == '{roi_for_corr}'")
    .join(dataset_info).groupby(["mouse_id", "day"])
    .agg(dict(y='mean'))
    .join(dataset_performance_rel)
    .join(dataset_info)
    .filter(["y", finemotor_for_corr, "g", "p"])
    .dropna()
)
df_corr.to_csv(p_figures / f'figure_2_correlation_data_{roi_for_corr}_{finemotor_for_corr}_wf{wf_param_id}.csv')
display(df_corr)

f, ax = plt.subplots(1, 2, figsize=(5.5*cm, 3*cm), sharex=True, sharey=True,
                     gridspec_kw=dict(wspace=0, hspace=0, top=0.95, bottom=0.2, left=0.2, right=0.95))

early = df_corr.query("p == 'Post Early'").dropna()
colors_early = [group_colors[g] for g in early["g"]]
from scipy.stats import spearmanr
rho, pval = spearmanr(early[finemotor_for_corr], early['y'])
ax[0].scatter(early[finemotor_for_corr], early['y'], label='Early', c=colors_early, alpha=.7, edgecolors='none')
ax[0].text(0.03, 0.97, f"$\\rho$={rho:.3f}\np={pval:.3f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[0].transAxes)
late = df_corr.query("p == 'Post Late' ").dropna()
colors_late = [group_colors[g] for g in late["g"]]
rho, pval = spearmanr(late[finemotor_for_corr], late['y'])
ax[1].scatter(late[finemotor_for_corr], late['y'], label='Late', c=colors_late, alpha=.7, edgecolors='none')
ax[1].text(0.03, 0.97, f"$\\rho$={rho:.3f}\np={pval:.3f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[1].transAxes)

ax[0].set_title("Early")
ax[1].set_title("Late")
ax[0].set_ylabel("Change in $\\Delta$F/F (%)")
ax[0].set_xlabel(xlabel)

# ax[0].set_xlabel("Lesion volume (mm$^3$)")
f.savefig(p_figures / f'figure_2_roi_vs_performance_{roi_for_corr}_{finemotor_for_corr}_wf{wf_param_id}.svg', dpi=600, transparent=True)
plt.show()

### Figure 2F, supplementary figure 2B - motor features

In [ ]:
roi_for_corr = "MOp_contra"
finemotor_for_corr = "bend_24_mean"
# stroke_group = "Stroke + training"
xlabel = "Change in limb bending (°)"

finemotor_mean = (
    dataset_finemotor_rel
    .groupby(["mouse_id", "day"])
    .agg({finemotor_for_corr: 'mean'})
)
display(finemotor_mean)

df_corr = (
    df_response_rois_for_stats
    .query(f"roi == '{roi_for_corr}'")
    .join(dataset_info).groupby(["mouse_id", "day"])
    .agg(dict(y='mean'))
    .join(finemotor_mean)
    .join(dataset_info)
    .filter(["y", finemotor_for_corr, "g", "p"])
    .dropna()
)
df_corr.to_csv(p_figures / f'figure_2_correlation_data_{roi_for_corr}_{finemotor_for_corr}_wf{wf_param_id}.csv')
display(df_corr)

f, ax = plt.subplots(1, 2, figsize=(5.5*cm, 3*cm), sharex=True, sharey=True,
                     gridspec_kw=dict(wspace=0, hspace=0, top=0.95, bottom=0.2, left=0.2, right=0.95))
early = df_corr.query("p == 'Post Early'").dropna()
colors_early = [group_colors[g] for g in early["g"]]
from scipy.stats import spearmanr
rho, pval = spearmanr(early[finemotor_for_corr], early['y'])
ax[0].scatter(early[finemotor_for_corr], early['y'], label='Early', c=colors_early, alpha=.7, edgecolors='none')
ax[0].text(0.03, 0.97, f"$\\rho$={rho:.3f}\np={pval:.3f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[0].transAxes)
late = df_corr.query("p == 'Post Late'").dropna()
colors_late = [group_colors[g] for g in late["g"]]
rho, pval = spearmanr(late[finemotor_for_corr], late['y'])
ax[1].scatter(late[finemotor_for_corr], late['y'], label='Late', c=colors_late, alpha=.7, edgecolors='none')
ax[1].text(0.03, 0.97, f"$\\rho$={rho:.3f}\np={pval:.3f}", fontsize=fontsize_small, ha='left', va='top', transform=ax[1].transAxes)

ax[0].set_title("Early")
ax[1].set_title("Late")
ax[0].set_ylabel("Change in $\\Delta$F/F (%)")
ax[0].set_xlabel(xlabel)
# ax[0].set_xlabel("Lesion volume (mm$^3$)")
f.savefig(p_figures / f'figure_2_roi_vs_finemotor_{roi_for_corr}_{finemotor_for_corr}_wf{wf_param_id}.svg', dpi=600, transparent=True)
plt.show()